# SERPdive

<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/tools/serpdive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[SERPdive](https://serpdive.com) is an AI search API built for agents: your agent asks a question, the API returns answer-ready web content that is extracted, cleaned, and sized for LLM consumption. On a [public, replayable 1,000-question benchmark](https://github.com/edendalexis/serpdive-benchmark), SERPdive runs at the same speed as Tavily, feeds your LLM 20.2% fewer tokens, and wins 60.7% of decided quality duels.

The `llama-index-tools-serpdive` package turns every search result into a `Document` whose text is the actual content of the page, with url, title and date in the metadata: ready for an agent to quote and cite, or for a RAG pipeline to index.

In [ ]:
%pip install llama-index-tools-serpdive llama-index-llms-openai

## Setup

Get a free API key (no card required) at [serpdive.com/dashboard/keys](https://serpdive.com/dashboard/keys) and set it as the `SERPDIVE_API_KEY` environment variable:

In [ ]:
import getpass
import os

if not os.environ.get("SERPDIVE_API_KEY"):
    os.environ["SERPDIVE_API_KEY"] = getpass.getpass("SERPdive API key:\n")

## Direct usage

The tool spec exposes a single function, `serpdive_search`. It takes a natural language query, in any language, and returns a list of `Document` objects:

In [1]:
from llama_index_tools_serpdive import SerpdiveToolSpec

tool_spec = SerpdiveToolSpec()

documents = tool_spec.serpdive_search("latest developments in solid state batteries")
for doc in documents[:3]:
    print(doc.metadata.get("title"), "|", doc.metadata["url"])
print()
print(documents[0].text[:300])

The biggest problem with solid-state batteries may finally be solved | ScienceDaily | https://www.sciencedaily.com/releases/2026/07/260710003533.htm
Solid-State Batteries 2026: Advances, Challenges & Applications | https://www.bonnenbatteries.com/solid-state-batteries-advances-challenges-future-use-cases/
Solid-State Batteries 2026-2036: Technology, Forecasts, Players: IDTechEx | https://www.idtechex.com/en/research-report/solid-state-batteries/1130

Yuwei Zhang, first author of the new publication and head of the group "Chemo-Mechanics of Battery Materials" at MPI-SusMat.
ScienceDaily, 10 July 2026. <www.sciencedaily.com/releases/2026/07/260710003533.htm>.
The biggest problem with solid-state batteries may finally be solved.
18, 2022  A new di


### Going deeper

The default model, `mako`, returns the fact-carrying sentences of each page, fast. For deep research, `moby` returns the full readable text of each page. `max_results` puts a hard cap (1-10) on delivered results; by default the engine picks its calibrated mix.

There is no country or locale parameter to configure: the language of the query picks where the search runs, automatically. Failed searches are never billed.

In [2]:
deep_spec = SerpdiveToolSpec(model="moby", max_results=3)

documents = deep_spec.serpdive_search("why are solid state batteries hard to manufacture at scale")
[doc.metadata["url"] for doc in documents]

['https://www.sciencedaily.com/releases/2026/07/260710003533.htm', 'https://www.bonnenbatteries.com/solid-state-batteries-advances-challenges-future-use-cases/', 'https://christopherchico.substack.com/p/solid-state-batteries-why-mass-production']

## Use within an agent

`to_tool_list()` converts the spec into function tools any LlamaIndex agent can call. The tool description tells the model when to search, so a plain question is enough to trigger it:

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

agent = FunctionAgent(
    tools=tool_spec.to_tool_list(),
    llm=OpenAI(model="gpt-5.5"),
)

response = await agent.run(
    "What is holding back mass production of solid state batteries? Cite your sources."
)
print(response)

## Links

- [API reference](https://serpdive.com/docs)
- [Public benchmark](https://github.com/edendalexis/serpdive-benchmark) (replayable end to end: same questions, same judge, your machine)
- [Package repo](https://github.com/serpdive/llama-index-tools-serpdive)